# Fase 4 — Modelado supervisado (Flujo B, parte 3)

| Campo | Valor |
|---|---|
| **Rol líder** | Científico de Datos |
| **Entrada** | `data/processed/chembl_clean.csv` |
| **Salidas** | Modelos `.pkl` en `outputs/chembl/models/` + `metrics_summary.csv` |
| **Doc de fase** | [`docs/analisis_proyecto/fases/fase4_modelado.md`](../../docs/analisis_proyecto/fases/fase4_modelado.md) |

Secciones 4 y 5 del Flujo B: clasificación y regresión.

> **Prerequisito:** ejecutar `fase2_limpieza.ipynb` para generar `chembl_clean.csv`.


## 0. Configuración


In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR

ROOT = Path.cwd().parent.parent if Path.cwd().name == "proyecto analisis de datos" else (
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analisis_proyecto.chembl_preprocessing import (
    ASSAY_FEATURE_COLS,
    build_supervised_matrix,
    evaluate_classification,
    evaluate_regression,
    get_available_feature_cols,
    load_bioactivity,
    train_test_split_by_group,
    train_test_split_rows,
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 120})

CLEAN_CSV = ROOT / "data" / "processed" / "chembl_clean.csv"
FIG_DIR = ROOT / "outputs" / "chembl" / "figures"
MODEL_DIR = ROOT / "outputs" / "chembl" / "models"
RESULTS_DIR = ROOT / "outputs" / "chembl" / "results"

for d in (FIG_DIR, MODEL_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

assert CLEAN_CSV.exists(), (
    f"No se encontró {CLEAN_CSV}. Ejecuta primero fase2_limpieza.ipynb"
)

df_clean = load_bioactivity(CLEAN_CSV)
feature_cols = get_available_feature_cols(df_clean)
print(f"Dataset limpio: {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas")
print(f"Features moleculares: {len(feature_cols)}")


## 4. Clasificación

**Objetivo:** `activity_class` (Active / Inactive)  
**Features:** descriptores moleculares (+ opcional contexto de ensayo en evaluación)  
**Splits evaluados:**
- **Por filas (80/20):** métrica estándar del curso
- **Por compuesto:** generalización a plaguicidas no vistos (misma molécula no en train y test)


In [ ]:
feature_cols = get_available_feature_cols(df_clean)
print(f"Features moleculares ({len(feature_cols)}): {feature_cols}")

X_all, y_class_all, groups_cls = build_supervised_matrix(
    df_clean,
    target_col="activity_class",
    numeric_cols=feature_cols,
    include_assay_features=False,
)

print(f"\nMuestras clasificación: {len(X_all):,}")
print(f"Compuestos únicos: {groups_cls.nunique()}")
print(y_class_all.value_counts())

# --- Split por filas (referencia curso) ---
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split_rows(
    X_all, y_class_all, test_size=0.2, random_state=RANDOM_STATE, stratify=True
)

rf_clf = RandomForestClassifier(
    n_estimators=100, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1
)
rf_clf.fit(X_train_c, y_train_c)

svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
])
svm_clf.fit(X_train_c, y_train_c)

cls_metrics = []
for name, model in [("RandomForest", rf_clf), ("SVM_RBF", svm_clf)]:
    scores = evaluate_classification(model, X_train_c, X_test_c, y_train_c, y_test_c)
    cls_metrics.append({
        "modelo": name,
        "tarea": "clasificacion",
        "split": "filas",
        "feature_set": "descriptores",
        **scores,
    })
    print(f"\n=== {name} (split filas) ===")
    print(f"Accuracy train: {scores['accuracy_train']:.4f} | test: {scores['accuracy_test']:.4f}")
    print(classification_report(y_test_c, model.predict(X_test_c)))

# --- Split por compuesto (generalización) ---
X_tr_g, X_te_g, y_tr_g, y_te_g = train_test_split_by_group(
    X_all, y_class_all, groups_cls, test_size=0.2, random_state=RANDOM_STATE
)
rf_group = RandomForestClassifier(
    n_estimators=100, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1
)
rf_group.fit(X_tr_g, y_tr_g)
scores_g = evaluate_classification(rf_group, X_tr_g, X_te_g, y_tr_g, y_te_g)
cls_metrics.append({
    "modelo": "RandomForest",
    "tarea": "clasificacion",
    "split": "compuesto",
    "feature_set": "descriptores",
    **scores_g,
})
print(f"\n=== RandomForest (split compuesto) ===")
print(f"Accuracy train: {scores_g['accuracy_train']:.4f} | test: {scores_g['accuracy_test']:.4f}")
print(f"Compuestos en test: {groups_cls.loc[X_te_g.index].nunique()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, model) in zip(axes, [("RandomForest", rf_clf), ("SVM_RBF", svm_clf)]):
    ConfusionMatrixDisplay.from_estimator(model, X_test_c, y_test_c, ax=ax, cmap="Blues")
    ax.set_title(f"Confusion matrix — {name} (filas)")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrices.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots()
le = LabelEncoder()
le.fit(["Inactive", "Active"])
y_bin = le.transform(y_test_c)
RocCurveDisplay.from_estimator(rf_clf, X_test_c, y_bin, name="RandomForest", ax=ax)
RocCurveDisplay.from_estimator(svm_clf, X_test_c, y_bin, name="SVM_RBF", ax=ax)
ax.set_title("ROC — test set (split filas)")
plt.tight_layout()
plt.savefig(FIG_DIR / "roc_curves.png", bbox_inches="tight")
plt.show()

joblib.dump(rf_clf, MODEL_DIR / "rf_classifier.pkl")
joblib.dump(svm_clf, MODEL_DIR / "svm_classifier.pkl")
with open(MODEL_DIR / "feature_cols.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)
print(f"Modelos guardados en {MODEL_DIR}")


## 5. Regresión

**Objetivo:** `pchembl_value`  
**Modelos:** SVR (RBF) y Random Forest Regressor  

Se comparan dos conjuntos de features y dos protocolos de split:

| Config | Features | Split | Interpretación |
|--------|----------|-------|----------------|
| A | Descriptores moleculares | Por filas | Referencia del curso (puede inflar R²) |
| B | Descriptores moleculares | Por compuesto | Generalización química real |
| C | Descriptores + ensayo | Por filas | Mejor ajuste (incluye diana/tipo de medida) |
| D | Descriptores + ensayo | Por compuesto | Evaluación más honesta |

**Métricas:** R² train/test, MAE, RMSE, scatter predicho vs real


In [ ]:
def _make_regressors():
    return {
        "SVR_RBF": Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(kernel="rbf")),
        ]),
        "RandomForest": RandomForestRegressor(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
        ),
    }


def run_regression_benchmark(
    df: pd.DataFrame,
    *,
    include_assay: bool,
    split_by: str,
    feature_set_label: str,
) -> tuple[list[dict], dict]:
    """Entrena SVR y RF; devuelve métricas y modelos ajustados."""
    X, y, groups = build_supervised_matrix(
        df,
        target_col="pchembl_value",
        numeric_cols=feature_cols,
        include_assay_features=include_assay,
    )
    if split_by == "compuesto":
        X_train, X_test, y_train, y_test = train_test_split_by_group(
            X, y, groups, test_size=0.2, random_state=RANDOM_STATE
        )
    else:
        X_train, X_test, y_train, y_test = train_test_split_rows(
            X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=False
        )

    rows = []
    fitted = {}
    for name, model in _make_regressors().items():
        model.fit(X_train, y_train)
        scores = evaluate_regression(model, X_train, X_test, y_train, y_test)
        rows.append({
            "modelo": name,
            "tarea": "regresion",
            "split": split_by,
            "feature_set": feature_set_label,
            "n_train": len(X_train),
            "n_test": len(X_test),
            **scores,
        })
        fitted[name] = (model, X_test, y_test)
    return rows, fitted


reg_configs = [
    (False, "filas", "descriptores"),
    (False, "compuesto", "descriptores"),
    (True, "filas", "descriptores+ensayo"),
    (True, "compuesto", "descriptores+ensayo"),
]

reg_metrics = []
fitted_models = {}
for include_assay, split_by, feat_label in reg_configs:
    rows, fitted = run_regression_benchmark(
        df_clean,
        include_assay=include_assay,
        split_by=split_by,
        feature_set_label=feat_label,
    )
    reg_metrics.extend(rows)
    fitted_models[(feat_label, split_by)] = fitted

reg_df = pd.DataFrame(reg_metrics)
display(reg_df.round(4))

# Scatter: config A (descriptores/filas) vs D (descriptores+ensayo/compuesto)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_specs = [
    ("descriptores", "filas", "RandomForest", "A — descriptores, split filas"),
    ("descriptores+ensayo", "compuesto", "RandomForest", "D — descriptores+ensayo, split compuesto"),
]
for ax, (feat, split_by, model_name, title) in zip(axes, plot_specs):
    model, X_test, y_test = fitted_models[(feat, split_by)][model_name]
    pred = model.predict(X_test)
    ax.scatter(y_test, pred, alpha=0.35, s=18)
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, "r--", lw=1)
    r2 = r2_score(y_test, pred)
    ax.set_xlabel("pChEMBL real")
    ax.set_ylabel("pChEMBL predicho")
    ax.set_title(f"{title}\nR² test = {r2:.3f}")

plt.tight_layout()
plt.savefig(FIG_DIR / "regression_pred_vs_real.png", bbox_inches="tight")
plt.show()

# Modelos finales para dashboard: mejor config en split filas (descriptores+ensayo)
svr_reg = fitted_models[("descriptores+ensayo", "filas")]["SVR_RBF"][0]
rf_reg = fitted_models[("descriptores+ensayo", "filas")]["RandomForest"][0]
joblib.dump(svr_reg, MODEL_DIR / "svr_regressor.pkl")
joblib.dump(rf_reg, MODEL_DIR / "rf_regressor.pkl")

# Guardar nombres de columnas del modelo enriquecido
X_full, _, _ = build_supervised_matrix(
    df_clean,
    target_col="pchembl_value",
    numeric_cols=feature_cols,
    include_assay_features=True,
)
with open(MODEL_DIR / "regression_feature_cols.json", "w", encoding="utf-8") as f:
    json.dump(list(X_full.columns), f, indent=2)

metrics_df = pd.DataFrame(cls_metrics + reg_metrics)
metrics_path = RESULTS_DIR / "metrics_summary.csv"
metrics_df.to_csv(metrics_path, index=False)
display(metrics_df.round(4))
print(f"\nMetricas guardadas: {metrics_path}")
print("\n=== Flujo B completado ===")


---
*Fase anterior:* [`fase3_eda.ipynb`](fase3_eda.ipynb)  
*Siguiente fase:* [`fase5_dashboard.ipynb`](fase5_dashboard.ipynb)
